In [1]:
import cv2
import torch
import os
from facenet_pytorch import MTCNN, InceptionResnetV1

c:\Users\MEHMET\anaconda3\envs\cv_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


database (Embeddings)

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"[*] Running on device: {device.upper()}")

mtcnn = MTCNN(keep_all=True, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

database = {}
images_folder = "images_recognition"
print("Loading database...")

for person_name in os.listdir(images_folder):
    person_path = os.path.join(images_folder, person_name)
    embeddings = []
        
    for img_file in os.listdir(person_path):
        img_path = os.path.join(person_path, img_file)
            
        img = cv2.imread(img_path)
        if img is None:
            continue
                
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        faces = mtcnn(img_rgb)
            
        if faces is not None:
            for face in faces:
                emb = resnet(face.unsqueeze(0).to(device))
                embeddings.append(emb.detach().cpu())
        
        if len(embeddings) > 0:
            mean_embedding = torch.mean(torch.stack(embeddings), dim=0)
            database[person_name] = mean_embedding
print("Database loaded!")

[*] Running on device: CUDA
Loading database...
Database loaded!


Face Recognition using FaceNet(MTCC,InceptionResnetV1) with camera

In [3]:
cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if not ret: break
    
    small_frame = cv2.resize(frame, (0, 0), fx=0.5, fy=0.5)
    img_rgb = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)
    
    boxes, probs = mtcnn.detect(img_rgb)
    faces = mtcnn(img_rgb)
    
    if boxes is not None and faces is not None:
        for i, face in enumerate(faces):
            box = boxes[i]
            
            x, y, x2, y2 = (box * 2).astype(int) 
            curr_emb = resnet(face.unsqueeze(0).to(device)).detach().cpu()
            
            min_dist = 0.85
            name = "Unknown"
            
            for person, db_emb in database.items():
                dist = (db_emb - curr_emb).norm().item()
                if dist < min_dist:
                    min_dist = dist
                    name = person
            
            color = (0, 255, 0) if name != "Unknown" else (0, 0, 255) 
            cv2.rectangle(frame, (x, y), (x2, y2), color, 2)
            cv2.putText(frame, name, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

    cv2.imshow("Face Recognition", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

Face Recognition using FaceNet(MTCC,InceptionResnetV1) with image

In [4]:
test_image_path = "test\\3.jpeg" 
frame = cv2.imread(test_image_path)

if frame is None:
    print(f"Error: Could not read image at {test_image_path}")
else:
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    boxes, probs = mtcnn.detect(img_rgb)
    faces = mtcnn(img_rgb) 
    
    if boxes is not None and faces is not None:
        for i, face in enumerate(faces):
            box = boxes[i]
            x, y, x2, y2 = box.astype(int) 
            
            curr_emb = resnet(face.unsqueeze(0).to(device)).detach().cpu()
            
            min_dist = 0.85
            name = "Unknown"
            
            for person, db_emb in database.items():
                dist = (db_emb - curr_emb).norm().item()
                if dist < min_dist:
                    min_dist = dist
                    name = person
            
            color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
            cv2.rectangle(frame, (x, y), (x2, y2), color, 2)
            cv2.putText(frame, name, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

    cv2.imshow("Static Face Recognition", frame)
    
    cv2.waitKey(0) 
    cv2.destroyAllWindows()